# Shop · RetailAction on Google Colab T4

**Delivery status (18 November 2026): prepared; GPU execution not yet performed.** Local preprocessing and synthetic checks are separate
from the T4 learning pilot. Training is blocked until all original Drive archives are verified.

This is a separate offline `take / put / touch` **clip-presence** baseline using publisher pose
observations. It does not overwrite MERL, localize action intervals, classify theft, or enable
Shop inference. Missing poses/time are masked. Some actual sequences exceed the dataset card's
32-frame description; bounded padding preserves observations without pretending they are 5 Hz.

Select **Runtime → Change runtime type → T4 GPU**. No paid upgrade is performed. Colab allocation
and session lifetime are not guaranteed. The runtime stays CPU-only on your Pop!_OS application.

Sources: [RetailAction](https://huggingface.co/datasets/standard-cognition/RetailAction/blob/main/README.md),
[license](https://huggingface.co/datasets/standard-cognition/RetailAction/blob/main/LICENSE),
[Colab limits](https://research.google.com/colaboratory/faq.html).

## 1. Runtime and experiment settings
Keep RUN_ID stable when resuming. Use a new RUN_ID for a new validation experiment. The final-test
registry is shared across runs to reject sequential reuse for model selection. Use only one active
evaluator across all Colab runtimes sharing this registry; local locks are not distributed. Test evaluation is a
separate opt-in cell after model/validation choices are settled; it is not needed to train.

In [ ]:
from pathlib import Path
import json, sys, shutil, subprocess, hashlib, platform, time
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/ShopliftingAI-Data')
RUN_ID = 'retailaction_t4_20261118_v1'
RUN_FULL_TRAINING = True
MODE = 'train'  # Use 'evaluate' only after validation/model choices are settled.
if MODE not in ('train', 'evaluate'):
    raise ValueError('MODE must be train or evaluate')

if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Select a Colab T4 GPU runtime. CPU and other-GPU fallback are disabled.')
if not hasattr(torch, 'amp') or not hasattr(torch.amp, 'GradScaler'):
    raise RuntimeError('This runtime needs modern CUDA-enabled PyTorch with torch.amp.')
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
       'cuda_build': torch.version.cuda, 'python': platform.python_version()})
# Do NOT install requirements-cpu.txt or the full application into this CUDA runtime.

## 2. Mount Drive and verify upload readiness
Authorize your own Google account in the Colab prompt. Change DRIVE_ROOT only if the selected
folder is mounted elsewhere. No public sharing or permissions are changed.

In [ ]:
READINESS_STATE = None
STAGING_STATE = None
import uuid
from google.colab import drive
drive.mount('/content/drive')
SETUP_ROOT = DRIVE_ROOT

RAW = DRIVE_ROOT / 'raw/retailaction'
STATUS = RAW / 'TRANSFER_STATUS.json'
status = json.loads(STATUS.read_text())
expected = {
 'train.tar': (26082080256, '272dcd90f2a12020dfe40542b54196dc36309a0e42298ff50f3c384956369ad0'),
 'validation.tar': (1937612800, '1287dc0b491f3eab5807d216bb96db2436da845ed0ea6e4ec69daf6182873836'),
 'test.tar': (3838385152, '26a376627f22df0702d1bb536dd073843d3720a6b4a1788d4d226513a3ac8a25'),
}
records = {v['name']: v for v in status.get('files', [])}
if status.get('state') != 'complete':
    raise RuntimeError('Upload/verification is incomplete. Stop here and rerun after completion.')
for name, (size, digest) in expected.items():
    row = records.get(name, {})
    if not (row.get('verified') is True and row.get('actual_bytes') == size
            and row.get('actual_sha256') == digest and (RAW/name).stat().st_size == size):
        raise RuntimeError('Missing or incompatible verified archive: '+name)
# Recheck current test bytes without parsing labels; the receipt alone is not immutable.
digest = hashlib.sha256()
with (RAW/'test.tar').open('rb') as source:
    for block in iter(lambda: source.read(8*1024*1024), b''): digest.update(block)
if digest.hexdigest() != expected['test.tar'][1]:
    raise RuntimeError('Current test archive no longer matches its verified receipt')
READINESS_STATE = {'root': str(DRIVE_ROOT), 'receipt': str(STATUS), 'token': uuid.uuid4().hex}
print('Receipt verified for all three archives; current test bytes verified. Train/validation current bytes are checked in staging.')

## 3. Load the reviewed code bundle
The bundle contains only the experiment and the existing causal-TCN source, with no credentials,
local environment, customer footage, database or MERL weights.

In [ ]:
BUNDLE_STATE = None
import zipfile, io
BUNDLE = DRIVE_ROOT / 'notebooks/shop_retailaction_t4_source.zip'
BUNDLE_SHA256 = 'b4310a4b056930ba576971f83af97343b9cfe3099d5981edcbb7ddeab6edaafa'
bundle_bytes = BUNDLE.read_bytes()
if hashlib.sha256(bundle_bytes).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError('Code bundle checksum mismatch; obtain the matching notebook and source bundle.')
CODE = Path('/content/shop-retailaction-code')
allowed = set(['ml/__init__.py', 'ml/retailaction/__init__.py', 'ml/retailaction/data.py', 'ml/retailaction/train.py', 'ml/retailaction/state.py', 'ml/retailaction/evaluate.py', 'ml/retailaction/README.md', 'packages/shop_core/shop_core/__init__.py', 'packages/shop_core/shop_core/temporal/__init__.py', 'packages/shop_core/shop_core/temporal/tcn.py'])
loaded_digest = globals().get('_SHOP_LOADED_BUNDLE_SHA256')
loaded_helpers = any(name == 'ml.retailaction' or name.startswith('ml.retailaction.') for name in sys.modules)
if loaded_helpers and loaded_digest != BUNDLE_SHA256:
    raise RuntimeError('An older/unverified experiment is imported. Restart the runtime and rerun this notebook.')
for package, origin in [('ml', CODE/'ml/__init__.py'),
                        ('shop_core', CODE/'packages/shop_core/shop_core/__init__.py')]:
    existing = sys.modules.get(package)
    if existing is not None and Path(getattr(existing, '__file__', '') or '/').resolve() != origin.resolve():
        raise RuntimeError('An older checkout is imported. Restart the Colab runtime.')
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    if set(archive.namelist()) != allowed:
        raise RuntimeError('Unexpected source bundle members')
    CODE.mkdir(exist_ok=True)
    archive.extractall(CODE)  # Exact known filenames, protected by bundle hash above.
sys.path[:0] = [str(CODE), str(CODE/'packages/shop_core')]
from ml.retailaction import data as verified_data
if Path(verified_data.__file__).resolve() != (CODE/'ml/retailaction/data.py').resolve():
    raise RuntimeError('Setup helper did not load from the verified source bundle; restart runtime.')
check_drive_gate = verified_data.check_drive_gate
check_drive_gate(STATUS)
_SHOP_LOADED_BUNDLE_SHA256 = BUNDLE_SHA256
BUNDLE_STATE = {'root': str(DRIVE_ROOT), 'sha256': BUNDLE_SHA256}
print('Source bundle verified:', BUNDLE_SHA256)

def verified_sources():
    if not BUNDLE_STATE or BUNDLE_STATE['root'] != str(DRIVE_ROOT) or BUNDLE_STATE['sha256'] != BUNDLE_SHA256:
        raise RuntimeError('Current source bundle verification has not passed.')
    if not READINESS_STATE or READINESS_STATE['root'] != str(DRIVE_ROOT):
        raise RuntimeError('Current Drive readiness has not passed.')

## 4. Stage training and validation archives
Use Colab's local disk for sequential archive reads. Files remain archived; the loader reads
metadata directly and validates that both videos exist. It does not extract arbitrary TAR paths.
Only training and validation are staged now; final-test labels stay untouched.

In [ ]:
STAGING_STATE = None
LOCAL = Path('/content/shop-retailaction-data')
LOCAL.mkdir(exist_ok=True)
from ml.retailaction.data import check_archive, sha256

def cached(name):
    target = LOCAL/name
    if target.exists():
        try:
            check_archive(target)
            return True
        except ValueError:
            target.unlink()  # Invalid cache owned by this notebook, not the Drive original.
    return False

def stage(names):
    global STAGING_STATE
    previous = STAGING_STATE
    STAGING_STATE = None
    verified_sources()
    verified = set()
    if previous and previous['root'] == str(DRIVE_ROOT) and previous['token'] == READINESS_STATE['token']:
        verified.update(previous['archives'])
    verified.difference_update(names)
    STAGING_STATE = {'root': str(DRIVE_ROOT), 'token': READINESS_STATE['token'],
                     'archives': sorted(verified)}
    # Only abandoned copies owned by this staging operation are removed.
    for name in names:
        (LOCAL/name).with_suffix('.tar.part').unlink(missing_ok=True)
    ready = {name: cached(name) for name in names}
    needed = sum(expected[name][0] for name in names if not ready[name])
    if shutil.disk_usage(LOCAL).free < needed + 2_000_000_000:
        raise RuntimeError('Insufficient local runtime disk space for safe staging')
    for name in names:
        if ready[name]:
            if sha256(RAW/name) != expected[name][1]:
                raise RuntimeError('Current Drive archive differs from the verified cache: '+name)
            verified.add(name)
            STAGING_STATE['archives'] = sorted(verified)
            continue
        target = LOCAL/name
        temporary = target.with_suffix('.tar.part')
        print('Staging', name, flush=True)
        shutil.copyfile(RAW/name, temporary)
        if temporary.stat().st_size != expected[name][0] or sha256(temporary) != expected[name][1]:
            temporary.unlink(missing_ok=True)
            raise RuntimeError('Copied archive checksum mismatch: '+name)
        temporary.replace(target)
        verified.add(name)
        STAGING_STATE['archives'] = sorted(verified)
    verified.update(names)
    STAGING_STATE = {'root': str(DRIVE_ROOT), 'token': READINESS_STATE['token'],
                     'archives': sorted(verified)}

from ml.retailaction.state import run_lock

def stage_final_inputs(full):
    verified_sources()
    with run_lock(full):
        if not (full/'completed.json').exists() or not (full/'best.pt').exists():
            raise RuntimeError('A completed validation run is required before final-only staging.')
        completion = json.loads((full/'completed.json').read_text())
        if completion.get('state') != 'complete' or completion.get('checkpoint_sha256') != sha256(full/'best.pt'):
            raise RuntimeError('Selected checkpoint is not the completed validation model.')
        # Free only notebook-owned, regenerable local caches. Drive originals stay persistent.
        for name in ('train.tar', 'validation.tar'):
            (LOCAL/name).unlink(missing_ok=True)
            (LOCAL/name).with_suffix('.tar.part').unlink(missing_ok=True)
        if STAGING_STATE:
            STAGING_STATE['archives'] = [name for name in STAGING_STATE['archives']
                                         if name not in ('train.tar','validation.tar')]
        stage(('test.tar',))

verified_sources()
if MODE == 'train':
    stage(('train.tar', 'validation.tar'))
    print('Current train/validation bytes verified. No test labels loaded.')
else:
    stage_final_inputs(DRIVE_ROOT/'checkpoints/retailaction_clip_tcn_v1'/RUN_ID/'full')
    print('Final-only mode: test cache staged without retaining development caches.')

## 5. T4 learning pilot
The loader audits all source metadata. It explicitly records excluded or mismatched samples,
keeps both views in one example, and blocks excessive overall or rare-class removal. The tiny
32-clip learning check runs 30 updates/epochs with finite gradients and decreasing loss required.
This is not an accuracy claim or a commercial model evaluation.

In [ ]:
from datetime import datetime, timezone
import uuid
from ml.retailaction.state import atomic_json

def run_paths(require_training=False):
    verified_sources()
    if require_training:
        if (not STAGING_STATE or STAGING_STATE['root'] != str(DRIVE_ROOT)
                or STAGING_STATE['token'] != READINESS_STATE['token']
                or not {'train.tar', 'validation.tar'}.issubset(STAGING_STATE['archives'])):
            raise RuntimeError('Verified training caches do not match the current root; rerun staging.')
    root = DRIVE_ROOT / 'checkpoints/retailaction_clip_tcn_v1' / RUN_ID
    return root, root/'pilot', root/'full'

RUN_ROOT, PILOT, FULL = run_paths()
RUN_ROOT.mkdir(parents=True, exist_ok=True)
environment = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
               'cuda_build': torch.version.cuda, 'python': platform.python_version(),
               'bundle_sha256': BUNDLE_STATE['sha256'], 'run_id': RUN_ID,
               'attempt_utc': datetime.now(timezone.utc).isoformat(), 'state': 'launch_attempt', 'mode': MODE}
if not (RUN_ROOT/'environment.json').exists():
    atomic_json(environment, RUN_ROOT/'environment.json')
atomic_json(environment, RUN_ROOT/'attempts'/('environment-'+uuid.uuid4().hex+'.json'))

def execute(arguments, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a') as log:
        process = subprocess.Popen(arguments, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            if process.wait() != 0:
                raise RuntimeError('Training/evaluation stopped; inspect '+str(log_path))
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
            raise
        finally:
            process.stdout.close()

def common_arguments():
    if MODE != 'train': raise RuntimeError('Training launch requested in evaluation-only mode.')
    run_paths(require_training=True)  # Recheck current setup for every training launch.
    return [sys.executable, '-u', str(CODE/'ml/retailaction/train.py'),
            '--archives', str(LOCAL), '--drive-status', READINESS_STATE['receipt']]

if MODE == 'train':
    command = common_arguments() + ['--out', str(PILOT), '--pilot', '--batch', '32']
    if (PILOT/'last.pt').exists(): command += ['--resume']
    execute(command, PILOT/'console.log')
    pilot = json.loads((PILOT/'pilot_report.json').read_text())
    if not pilot.get('passed'):
        raise RuntimeError('Pilot failed; full training remains blocked')
    print('Learning pilot passed. Review data_report.json for coverage and rare-class exclusions.')
else:
    print('Evaluation-only mode: learning pilot skipped; the completed checkpoint is required.')

## 6. Full T4 experiment and validation selection
Best and last checkpoints, optimizer/scaler state, history and data report are saved separately
on Drive every epoch. No final-test data participates in class weighting, learning or selection.
If Colab disconnects, reconnect a T4, keep RUN_ID, and rerun; compatible checkpoints resume.

In [ ]:
RUN_ROOT, PILOT, FULL = run_paths()
if MODE == 'train' and RUN_FULL_TRAINING:
    command = common_arguments() + ['--out', str(FULL), '--pilot-report', str(PILOT/'pilot_report.json'),
                        '--epochs', '60', '--patience', '10', '--batch', '128']
    if (FULL/'last.pt').exists(): command += ['--resume']
    execute(command, FULL/'console.log')
else:
    print('No full training requested in this mode/configuration.')

## 7. Inspect actual learning curves
AP is per paired clip and averaged only over supported classes; inspect support counts too.
It is not MERL frame mAP, temporal-localization AP, accuracy or a theft probability.

In [ ]:
RUN_ROOT, PILOT, FULL = run_paths()
if (FULL/'history.json').exists():
    import matplotlib.pyplot as plt
    history = json.loads((FULL/'history.json').read_text())
    epochs = [r['epoch'] for r in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(epochs, [r['train_loss'] for r in history], color='#087F78')
    axes[0].set(xlabel='Epoch', ylabel='Training BCE loss', title='Actual training loss')
    axes[1].plot(epochs, [r['validation']['clip_map_supported_classes'] for r in history], color='#142B3B')
    axes[1].set(xlabel='Epoch', ylabel='Validation clip mean AP', title='Actual validation selection metric')
    for axis in axes: axis.grid(alpha=0.2)
    fig.tight_layout(); plt.show()
    print(history[-1]['validation'])
else:
    print('No full-run history yet; no graph or result is fabricated.')

## 8. Frozen final test — run once after validation experiments are settled
This cell is disabled while exploring models. Set MODE='evaluate' only after deciding the
final model from validation. With one active evaluator, the registry rejects another checkpoint reusing this final
test for selection. Do not run parallel evaluators in separate Colab runtimes. It is an experiment guard, not a claim of production readiness.

In [ ]:
RUN_ROOT, PILOT, FULL = run_paths()
if MODE == 'evaluate':
    if not (FULL/'best.pt').exists() or not (FULL/'completed.json').exists():
        raise RuntimeError('A completed full validation experiment is required')
    completion = json.loads((FULL/'completed.json').read_text())
    if completion.get('state') != 'complete' or completion.get('checkpoint_sha256') != sha256(FULL/'best.pt'):
        raise RuntimeError('Selected checkpoint does not match the completed validation run')
    stage_final_inputs(FULL)
    target = LOCAL/'test.tar'
    EVALUATION = DRIVE_ROOT/'evaluation/retailaction_clip_tcn_v1'/RUN_ID
    command = [sys.executable, '-u', str(CODE/'ml/retailaction/evaluate.py'),
               '--archive', str(target), '--checkpoint', str(FULL/'best.pt'),
               '--out', str(EVALUATION), '--dataset-root', str(DRIVE_ROOT)]
    execute(command, EVALUATION/'console.log')
else:
    print('Final test remains untouched. Select/freeze the model on validation first.')

## 9. Deployment boundary
No cell enables the model in Shop. Next gates are a CPU network smoke test, compatible pose
extraction and subject assignment, representative target-store evaluation, full pipeline latency,
calibration if probabilities are shown, and commercial-rights review. CPU timing on the Colab
host would not establish Pop!_OS performance. Keep existing MERL artifacts and runtime toggles.